# 🏦 World Bank 2015 — Financial & Economic Analysis

**Author:** Anurag Das  
**Dataset:** 2015 World Bank Data (264 countries, 11 indicators)  
**Focus:** Banking readiness, lending potential, stock market correlation, and financial development across world regions

---

### 📌 Financial Questions We Answer:
1. Which regions have the strongest GDP base for financial market development?
2. Does internet penetration predict financial access and digital banking readiness?
3. Which regions are most export-driven — potential trade finance opportunities?
4. Which countries show the best profile for lending and credit growth?
5. How does population size relate to potential retail banking market size?
6. What is India's financial market potential compared to global peers?
7. Which indicators are most correlated with financial development?

---

### 🗺️ How to Use This Notebook
- Run cells **from top to bottom** — each cell builds on the one before it
- Press **Shift + Enter** to run a cell and move to the next one
- All charts are automatically saved to the `images/` folder
- Look for `# NOTE:` comments in the code — those are the settings you can safely change

---
## 📦 Step 1: Import Libraries

Before we can do any analysis, we need to load the tools (called **libraries**) that Python uses.
Think of this like opening all your apps before starting work.

| Library | What it does |
|---|---|
| `pandas` | Reads and works with tables of data (like Excel in Python) |
| `numpy` | Does fast math and number calculations |
| `matplotlib` | Creates charts and graphs |
| `seaborn` | Makes charts look more polished and statistical |
| `warnings` | Hides unimportant warning messages so output stays clean |

We also set up some **color and size settings** here so all our charts look consistent.
You can change `PALETTE`, `ACCENT_COLOR`, or `HIGHLIGHT` to customize the look.

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Plot Style ───────────────────────────────────────────────────────────────
# NOTE: You can change the color palette here to match your preferred theme
PALETTE       = 'Blues_d'        # Change this to any seaborn palette
ACCENT_COLOR  = '#1f4e79'        # Used for single-color charts
HIGHLIGHT     = '#e63946'        # Used to highlight key countries (e.g. India)
FIG_SIZE_BAR  = (12, 6)
FIG_SIZE_SCATTER = (10, 6)
FIG_SIZE_HEAT = (10, 8)

sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Libraries loaded successfully')

---
## 📂 Step 2: Load & Inspect the Dataset

Here we load the raw data from the Excel file into a **DataFrame** — that's what pandas calls a table of data.

- `pd.read_excel(...)` reads the `.xls` file, just like opening it in Excel
- `df.shape` tells us the number of rows (countries) and columns (indicators)
- `df.head()` shows the first 5 rows so we can see what the data looks like

> 📁 **Important:** Make sure the data file is saved at `../data/2015 World Bank data by nation and region.xls`  
> If you moved it, update the `DATA_PATH` variable below.

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
# NOTE: Update DATA_PATH if you move the file
DATA_PATH = '../data/2015 World Bank data by nation and region.xls'

df = pd.read_excel(DATA_PATH, engine='xlrd')

print(f'Shape: {df.shape}')
print(f'\nColumns:\n{df.columns.tolist()}')
df.head()

---
## 🧹 Step 3: Clean & Rename Columns

Raw data column names are often long and messy. We **rename** them to short, simple names so they're easier to use in code.

For example: `'GDP per Capita'` becomes `'gdp_per_capita'` — shorter and easier to type.

We also **drop rows** where the country name or region is missing — those rows can't be grouped or charted meaningfully.

After this step, `df` is our clean, analysis-ready table.

In [ ]:
# ── Column Mapping ────────────────────────────────────────────────────────────
# NOTE: Update this dictionary if your column names are different
COLUMN_MAP = {
    'Country Name'             : 'country',
    'Region'                   : 'region',
    'Income Group'             : 'income_group',
    'GDP (billions)'           : 'gdp_billions',
    'GDP per Capita'           : 'gdp_per_capita',
    'Internet Users (%)'       : 'internet_pct',
    'Life Expectancy'          : 'life_expectancy',
    'Female Literacy Rate (%)'  : 'female_literacy',
    'Population'               : 'population',
    'Exports (% of GDP)'       : 'exports_pct_gdp',
}

# Rename only columns that exist in the dataset
df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df.columns}, inplace=True)

# ── Drop rows with no country or region ──────────────────────────────────────
df.dropna(subset=['country', 'region'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Clean dataset: {df.shape[0]} countries')
df.head()

---
## 🔍 Step 4: Missing Data Overview

Real-world datasets always have gaps — some countries didn't report every indicator in 2015.
This step **visualizes the missing data** so we know which columns have gaps before we start analyzing.

The heatmap shows:
- **Blue cells** = that value is missing for that country
- **White cells** = data is present

We also print a count of missing values per column — columns with many missing values need to be handled carefully in later steps.

In [ ]:
# ── Missing Value Heatmap ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

missing = df.isnull()
sns.heatmap(missing, cbar=False, yticklabels=False,
            cmap='Blues', ax=ax)

ax.set_title('Missing Data Map (Blue = Missing)', fontsize=14, fontweight='bold')
ax.set_xlabel('Columns', fontsize=11)

plt.tight_layout()
plt.savefig('../images/missing_data.png', bbox_inches='tight')
plt.show()

# Summary
print('\nMissing values per column:')
print(df.isnull().sum().sort_values(ascending=False))

---
## ❓ Q1: Which Regions Have the Strongest GDP Base for Financial Market Development?

**What is GDP?**  
GDP (Gross Domestic Product) is the total value of everything a country produces in a year — it measures the size of an economy.

**Why does this matter for finance?**  
Larger economies have bigger stock markets, more banks, and more opportunities for investment. A region's total GDP tells us where the major financial hubs are.

**What the code does:**
1. Groups all countries by region using `groupby('region')`
2. Adds up (`sum()`) the GDP of all countries in each region
3. Sorts from highest to lowest
4. Draws a horizontal bar chart so we can easily compare regions

> 💡 **What to look for:** The longest bar = the region with the deepest financial markets and most developed banking sector.

In [ ]:
# ── GDP by Region ──────────────────────────────────────────────────────────────
gdp_region = (
    df.groupby('region')['gdp_billions']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

fig, ax = plt.subplots(figsize=FIG_SIZE_BAR)

bars = ax.barh(gdp_region['region'], gdp_region['gdp_billions'],
               color=sns.color_palette(PALETTE, len(gdp_region)))

# Add value labels
for bar, val in zip(bars, gdp_region['gdp_billions']):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}B', va='center', fontsize=9)

ax.set_title('Total GDP by Region (2015) — Financial Market Depth Proxy',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total GDP (USD Billions)', fontsize=11)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}B'))

plt.tight_layout()
plt.savefig('../images/gdp_by_region.png', bbox_inches='tight')
plt.show()

print('\n💡 Insight: Regions with higher GDP typically have deeper stock markets,'
      ' more active lending, and higher banking penetration.')

---
## ❓ Q2: Does Internet Penetration Predict Digital Banking & Financial Access?

**What is internet penetration?**  
The percentage of a country's population that uses the internet.

**Why does this matter for finance?**  
You can't use mobile banking, digital payments, or fintech apps without internet. Countries with high internet access are ready for digital financial services — neobanks, online lending, e-wallets, etc.

**What the code does:**
1. Creates a **scatter plot** — each dot is one country, positioned by its internet usage (x-axis) and GDP per capita (y-axis)
2. Highlights one country (default: India) in red
3. Draws a **trend line** to show the overall direction (does more internet = higher GDP?)
4. Calculates the **correlation coefficient** (a number from -1 to +1) — closer to +1 means a strong positive link

> 💡 **What to look for:** If the trend line goes up-right, internet access and wealth are linked. A correlation above 0.7 is considered strong.

> 🔧 **To spotlight a different country**, change `HIGHLIGHT_COUNTRY = 'India'` to any country name.

In [ ]:
# ── GDP vs Internet Usage Scatter ─────────────────────────────────────────────
# NOTE: You can change HIGHLIGHT_COUNTRY to any country you want to spotlight
HIGHLIGHT_COUNTRY = 'India'

clean = df.dropna(subset=['gdp_per_capita', 'internet_pct'])

fig, ax = plt.subplots(figsize=FIG_SIZE_SCATTER)

# All countries
ax.scatter(clean['internet_pct'], clean['gdp_per_capita'],
           alpha=0.5, color=ACCENT_COLOR, edgecolors='white', linewidths=0.5,
           label='Countries')

# Highlight country
hl = clean[clean['country'] == HIGHLIGHT_COUNTRY]
if not hl.empty:
    ax.scatter(hl['internet_pct'], hl['gdp_per_capita'],
               color=HIGHLIGHT, s=150, zorder=5, label=HIGHLIGHT_COUNTRY)
    ax.annotate(HIGHLIGHT_COUNTRY,
                xy=(hl['internet_pct'].values[0], hl['gdp_per_capita'].values[0]),
                xytext=(10, 10), textcoords='offset points', fontsize=9,
                color=HIGHLIGHT)

# Trend line
m, b = np.polyfit(clean['internet_pct'], clean['gdp_per_capita'], 1)
x_line = np.linspace(clean['internet_pct'].min(), clean['internet_pct'].max(), 100)
ax.plot(x_line, m * x_line + b, '--', color='gray', alpha=0.7, label='Trend')

ax.set_title('Internet Access vs GDP per Capita — Digital Banking Readiness',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Internet Users (%)', fontsize=11)
ax.set_ylabel('GDP per Capita (USD)', fontsize=11)
ax.legend()

plt.tight_layout()
plt.savefig('../images/gdp_vs_internet.png', bbox_inches='tight')
plt.show()

corr = clean['gdp_per_capita'].corr(clean['internet_pct'])
print(f'\n📈 Correlation (GDP per Capita vs Internet Usage): {corr:.3f}')
print('💡 Insight: High correlation → internet access is a strong predictor of'
      ' financial services adoption and digital lending potential.')

---
## ❓ Q3: Which Regions Are Most Export-Driven — Trade Finance Opportunities?

**What are exports?**  
Goods and services a country sells to other countries. "Exports as % of GDP" tells us how dependent a country's economy is on selling to the world.

**Why does this matter for finance?**  
Countries that export a lot need financial tools to support international trade — things like **letters of credit** (guarantees of payment), **trade loans**, **currency exchange (FX)**, and **export insurance**. High export regions = big demand for trade banking services.

**What the code does:**
1. Groups countries by region
2. Calculates the **average** exports-as-%-of-GDP for each region (using `.mean()`)
3. Draws a vertical bar chart, highlighting the top region in red

> 💡 **What to look for:** The tallest bar = the region most dependent on trade = the strongest opportunity for trade finance products.

In [ ]:
# ── Exports as % of GDP by Region ─────────────────────────────────────────────
exports_region = (
    df.groupby('region')['exports_pct_gdp']
      .mean()
      .sort_values(ascending=False)
      .reset_index()
)

fig, ax = plt.subplots(figsize=FIG_SIZE_BAR)

colors = [HIGHLIGHT if i == 0 else ACCENT_COLOR for i in range(len(exports_region))]
bars = ax.bar(exports_region['region'], exports_region['exports_pct_gdp'],
              color=colors, edgecolor='white')

for bar, val in zip(bars, exports_region['exports_pct_gdp']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=9)

ax.set_title('Average Exports as % of GDP by Region — Trade Finance Potential',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Exports (% of GDP)', fontsize=11)
ax.set_xticklabels(exports_region['region'], rotation=30, ha='right')

plt.tight_layout()
plt.savefig('../images/exports_by_region.png', bbox_inches='tight')
plt.show()

print('\n💡 Insight: Regions with high exports-to-GDP ratios are strong candidates'
      ' for trade finance, export credit insurance, and FX hedging products.')

---
## ❓ Q4: Which Countries Show the Best Profile for Lending & Credit Growth?

**What is lending potential?**  
A score we build to estimate how attractive a country is for banks and lenders to offer credit.

**Why does this matter for finance?**  
Banks make money by giving loans. The best lending markets have lots of people to lend to, people wealthy enough to repay, and an educated population that understands financial products.

**How is the score calculated?**  
We combine 3 indicators into a single score (0 to 1):
- **Population (40%)** — more people = more potential borrowers
- **GDP per capita (35%)** — wealthier people = more likely to qualify and repay
- **Female literacy rate (25%)** — education correlates with financial literacy

Each indicator is first **normalized** (scaled to 0–1) so they're comparable regardless of their original units.

> 🔧 **To adjust the score weights**, change `WEIGHT_POPULATION`, `WEIGHT_GDP_CAPITA`, and `WEIGHT_LITERACY`. They must add up to 1.0.

In [ ]:
# ── Lending Potential Score ───────────────────────────────────────────────────
# NOTE: You can adjust the weights below to change how the score is calculated
# Weights: population (market size) + gdp_per_capita (affordability) + literacy (financial readiness)

WEIGHT_POPULATION   = 0.40   # How much weight to give population size
WEIGHT_GDP_CAPITA   = 0.35   # How much weight to give GDP per capita
WEIGHT_LITERACY     = 0.25   # How much weight to give female literacy rate

lending_df = df.dropna(subset=['population', 'gdp_per_capita', 'female_literacy']).copy()

# Normalize each metric to 0–1
for col in ['population', 'gdp_per_capita', 'female_literacy']:
    col_min = lending_df[col].min()
    col_max = lending_df[col].max()
    lending_df[f'{col}_norm'] = (lending_df[col] - col_min) / (col_max - col_min)

lending_df['lending_score'] = (
    WEIGHT_POPULATION * lending_df['population_norm'] +
    WEIGHT_GDP_CAPITA * lending_df['gdp_per_capita_norm'] +
    WEIGHT_LITERACY   * lending_df['female_literacy_norm']
)

# Top 15 countries
top_lending = lending_df.nlargest(15, 'lending_score')[['country', 'lending_score']]

fig, ax = plt.subplots(figsize=FIG_SIZE_BAR)

colors = [HIGHLIGHT if c == HIGHLIGHT_COUNTRY else ACCENT_COLOR
          for c in top_lending['country']]
ax.barh(top_lending['country'], top_lending['lending_score'],
        color=colors, edgecolor='white')

ax.set_title('Top 15 Countries by Lending Potential Score (2015)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Lending Potential Score (0–1)', fontsize=11)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../images/lending_potential.png', bbox_inches='tight')
plt.show()

print('\nTop 10 Lending Markets:')
print(top_lending.head(10).to_string(index=False))

---
## ❓ Q5: Population vs Internet Access — Retail Banking Market Size

**What is retail banking?**  
Everyday banking services for individuals — savings accounts, debit cards, personal loans, mobile payments.

**Why does this matter for finance?**  
The ideal target for a digital bank is a country with **lots of people** (big market) AND **growing internet access** (can actually reach them digitally). A country with a huge population but low internet is an *untapped* future market — as internet grows, so does the opportunity.

**What the code does:**
1. Picks the top 10 most populated countries
2. Creates a **dual-axis chart** — blue bars show population size (left axis), red line shows internet usage % (right axis)
3. This lets us compare both metrics side by side on one chart

> 💡 **What to look for:** Countries where the bar is tall (big population) but the line sits low (low internet) — those are the largest untapped digital banking markets.

> 🔧 **To show more countries**, change `TOP_N = 10` to a higher number.

In [ ]:
# ── Top 10 Populous Countries vs Internet Access ───────────────────────────────
# NOTE: Change TOP_N to show more or fewer countries
TOP_N = 10

top_pop = (
    df.dropna(subset=['population', 'internet_pct'])
      .nlargest(TOP_N, 'population')
      [['country', 'population', 'internet_pct']]
)

fig, ax1 = plt.subplots(figsize=FIG_SIZE_BAR)
ax2 = ax1.twinx()

x = range(len(top_pop))
bars = ax1.bar(x, top_pop['population'] / 1e6, color=ACCENT_COLOR,
               alpha=0.7, label='Population (M)')
line = ax2.plot(x, top_pop['internet_pct'], 'o-', color=HIGHLIGHT,
                linewidth=2, markersize=7, label='Internet (%)')

ax1.set_xticks(x)
ax1.set_xticklabels(top_pop['country'], rotation=30, ha='right')
ax1.set_ylabel('Population (Millions)', fontsize=11)
ax2.set_ylabel('Internet Users (%)', fontsize=11, color=HIGHLIGHT)
ax1.set_title(f'Top {TOP_N} Populous Countries: Population vs Internet Access\n— Retail Banking Market Sizing',
              fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../images/population_vs_internet.png', bbox_inches='tight')
plt.show()

print('\n💡 Insight: Countries with high population AND high internet penetration'
      ' represent the largest addressable markets for retail banking and digital lending.')

---
## ❓ Q6: India's Financial Market Position vs Global Peers

**What are we doing here?**  
We benchmark India against the **world average** across 5 key financial indicators.
This tells us where India is ahead of the world, and where it still has room to grow.

**The 5 indicators we compare:**
| Indicator | Why it matters financially |
|---|---|
| GDP per Capita | Individual wealth — ability to save and borrow |
| Internet (%) | Digital banking readiness |
| Life Expectancy | Proxy for health and stability of the economy |
| Female Literacy (%) | Financial literacy — ability to use banking products |
| Exports (% GDP) | Trade finance opportunity |

**What the code does:**
- Grey bars = world average for each metric
- Red bars = India's value
- The printout shows `▲` where India is above average and `▼` where it's below

> 🔧 **To benchmark a different country**, change `BENCHMARK_COUNTRY = 'India'` to any country name.

In [ ]:
# ── India vs Global Averages ───────────────────────────────────────────────────
# NOTE: Change BENCHMARK_COUNTRY to compare any country
BENCHMARK_COUNTRY = 'India'

METRICS = ['gdp_per_capita', 'internet_pct', 'life_expectancy',
           'female_literacy', 'exports_pct_gdp']
METRIC_LABELS = ['GDP per Capita', 'Internet (%)', 'Life Expectancy',
                 'Female Literacy (%)', 'Exports (% GDP)']

world_avg = df[METRICS].mean()
country_row = df[df['country'] == BENCHMARK_COUNTRY][METRICS]

if country_row.empty:
    print(f'⚠️ Country "{BENCHMARK_COUNTRY}" not found. Check spelling.')
else:
    country_vals = country_row.values[0]

    x = np.arange(len(METRICS))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width/2, world_avg.values, width, label='World Average',
           color='#adb5bd', edgecolor='white')
    ax.bar(x + width/2, country_vals, width, label=BENCHMARK_COUNTRY,
           color=HIGHLIGHT, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(METRIC_LABELS, rotation=15, ha='right')
    ax.set_title(f'{BENCHMARK_COUNTRY} vs World Average — Financial Benchmarking',
                 fontsize=13, fontweight='bold')
    ax.legend()

    plt.tight_layout()
    plt.savefig('../images/country_benchmark.png', bbox_inches='tight')
    plt.show()

    print(f'\n{BENCHMARK_COUNTRY} vs World Average:')
    for label, world, country in zip(METRIC_LABELS, world_avg.values, country_vals):
        diff = country - world
        arrow = '▲' if diff > 0 else '▼'
        print(f'  {label:<25}: {BENCHMARK_COUNTRY}={country:.1f} | World={world:.1f} {arrow} {abs(diff):.1f}')

---
## ❓ Q7: Correlation Heatmap — Which Indicators Drive Financial Development?

**What is correlation?**  
Correlation measures how closely two things move together. It's a number between -1 and +1:
- **+1.0** = they move perfectly together (when one goes up, the other always goes up)
- **0.0** = no relationship at all
- **-1.0** = they move in opposite directions

**Why does this matter for finance?**  
If we know which factors (internet access, literacy, GDP) move together with financial development, banks and investors can use **cheap-to-measure proxies** to predict where their services will grow next — without having direct banking data for every country.

**What the code does:**
1. Calculates the correlation between every pair of indicators
2. Displays the result as a **color-coded grid (heatmap)**
   - 🟢 Green = strong positive correlation
   - 🔴 Red = strong negative correlation
   - ⬜ White/Yellow = weak or no correlation
3. Only the lower triangle is shown (the upper half is a mirror image, so we hide it)
4. Prints the top correlations with GDP per Capita specifically

> 💡 **What to look for:** Which pairs of indicators are dark green? Those are the strongest relationships in the dataset.

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────────
CORR_COLS = ['gdp_billions', 'gdp_per_capita', 'internet_pct',
             'life_expectancy', 'female_literacy', 'population', 'exports_pct_gdp']

CORR_LABELS = ['GDP (Total)', 'GDP per Capita', 'Internet (%)',
               'Life Expectancy', 'Female Literacy', 'Population', 'Exports (% GDP)']

corr_data = df[CORR_COLS].dropna()
corr_matrix = corr_data.corr()
corr_matrix.columns = CORR_LABELS
corr_matrix.index   = CORR_LABELS

fig, ax = plt.subplots(figsize=FIG_SIZE_HEAT)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            vmin=-1, vmax=1, ax=ax)

ax.set_title('Correlation Matrix — Key Financial Development Indicators (2015)',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations with GDP per capita
print('\nTop correlations with GDP per Capita (financial strength proxy):')
gdp_corr = corr_matrix['GDP per Capita'].drop('GDP per Capita').sort_values(ascending=False)
print(gdp_corr)

---
## 📋 Summary: Key Financial Insights

This final cell auto-generates a summary using the variables we calculated in the cells above.
It pulls the top results from each analysis and prints them together in one clean report.

> ⚠️ **This cell uses results from earlier cells** — make sure you've run all previous cells first, otherwise you'll get a `NameError`.

In [ ]:
# ── Auto-generated Summary ────────────────────────────────────────────────────
top_gdp_region   = gdp_region.iloc[0]['region']
top_export_region = exports_region.iloc[0]['region']
top_lending_country = top_lending.iloc[0]['country']

print('=' * 60)
print('  📊 FINANCIAL ANALYSIS SUMMARY — WORLD BANK 2015')
print('=' * 60)
print(f'\n  🏦 Strongest GDP Region (stock market base): {top_gdp_region}')
print(f'  📦 Highest Export-Driven Region (trade finance): {top_export_region}')
print(f'  💳 Top Lending Potential Country: {top_lending_country}')
print(f'  📱 GDP-Internet Correlation: {corr:.3f} (digital banking readiness)')
print(f'\n  🇮🇳 {BENCHMARK_COUNTRY}: Large population + growing internet = ')
print(f'      high-potential market for retail banking & fintech lending')
print('\n' + '=' * 60)

---

## 🔧 How to Extend This Project

| What to add | How |
|---|---|
| Real stock market data | Use `yfinance` or World Bank API for market cap data |
| Banking penetration data | Add 'Account ownership (%)' from World Bank FINDEX |
| Credit-to-GDP ratio | Merge with IMF Financial Stability data |
| Time series analysis | Load multiple years (2010–2023) and plot trends |
| Machine learning | Predict lending risk scores using regression |

---
*Project by Anurag Das | World Bank 2015 Dataset*